In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import re
from pathlib import Path
import string
import pickle

from functools import partial

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression  # TODO we can also use this
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [3]:
from rital.data import (
    load_presidents,
    load_presidents_unseen,
    load_movies,
    load_movies_unseen,
)
from rital.preprocessing import preprocess

%load_ext autoreload
%autoreload 2

## Data Loading

In [4]:
x_presidents, y_presidents = load_presidents()
x_presidents_unseen = load_presidents_unseen()

# SVM on TF-IDF

In [ ]:
# Data
X_train = x_presidents
y_train = y_presidents

# CV splitter
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline
model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
        ("classifier", LinearSVC()),
    ]
)

# Hyperparameter grid
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 5],
    "tfidf__max_df": [0.9, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "classifier__C": [0.01, 0.1, 1, 10],
}

# Grid search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",  # or "accuracy"
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_}")

best_model = grid.best_estimator_

y_pred = cross_val_predict(best_model, X_train, y_train, cv=cv)

print("Classification report:")
print(classification_report(y_train, y_pred))

Fitting 5 folds for each of 96 candidates, totalling 480 fits
